# Tiền xử lý dữ liệu bệnh & triệu chứng

Notebook này **chỉ xử lý 2 file dữ liệu** trong thư mục `01_data`:
- `benh_trieuchung.csv` — mỗi dòng là một ca bệnh với các triệu chứng.
- `trongso_mucdo_nghiemtrong.csv` — trọng số (mức độ nặng) của từng triệu chứng.

**Mục tiêu:** biến dữ liệu chữ thành **bảng số** để máy học được, rồi lưu ra `features.csv`.

> Việc xử lý câu mô tả của bệnh nhân (NLP) nằm ở notebook riêng: `xu_ly_input_nlp.ipynb`.


## Bước 1. Đọc 2 file dữ liệu

In [1]:
import pandas as pd

# Đọc dữ liệu bệnh - triệu chứng
benh = pd.read_csv("../01_data/benh_trieuchung.csv")
print("Số ca bệnh:", len(benh), "| Số bệnh khác nhau:", benh["Benh"].nunique())
benh.head()


Số ca bệnh: 2100 | Số bệnh khác nhau: 14


,Benh,TrieuChung_1,TrieuChung_2,TrieuChung_3,TrieuChung_4,TrieuChung_5,TrieuChung_6,TrieuChung_7,TrieuChung_8
0,Cảm cúm,sốt cao,mệt mỏi,NaN,NaN,NaN,NaN,NaN,NaN
1,Hen suyễn,thở khò khè,đau ngực,khó thở,NaN,NaN,NaN,NaN,NaN
2,Viêm họng,đau họng,khó nuốt,NaN,NaN,NaN,NaN,NaN,NaN
3,Viêm dạ dày,buồn nôn,đau bụng,ợ chua,đầy hơi,NaN,NaN,NaN,NaN
4,Hen suyễn,đau ngực,thở khò khè,ho,khó thở,NaN,NaN,NaN,NaN


In [2]:
# Đọc bảng trọng số mức độ nghiêm trọng
trongso = pd.read_csv("../01_data/trongso_mucdo_nghiemtrong.csv")
print("Số triệu chứng có trọng số:", len(trongso))
trongso.head()


Số triệu chứng có trọng số: 37


,TrieuChung,TrongSo
0,sốt cao,6
1,sốt,4
2,sốt nhẹ,3
3,đau đầu,4
4,đau nhức cơ,3


## Bước 2. Tạo "từ điển" trọng số

Đổi bảng trọng số thành một `dict` để tra cứu nhanh: tên triệu chứng → mức độ nặng.


In [3]:
trong_so = dict(zip(trongso["TrieuChung"], trongso["TrongSo"]))
ds_trieuchung = list(trongso["TrieuChung"])   # danh sách triệu chứng (thứ tự cột)
print("Ví dụ:", {k: trong_so[k] for k in ds_trieuchung[:5]})


Ví dụ: {'sốt cao': 6, 'sốt': 4, 'sốt nhẹ': 3, 'đau đầu': 4, 'đau nhức cơ': 3}


## Bước 3. Mã hóa one-hot có trọng số

Mỗi **triệu chứng là một cột**. Với mỗi ca bệnh:
- Nếu **có** triệu chứng đó → điền **trọng số** của nó.
- Nếu **không có** → điền 0.

Như vậy mỗi ca bệnh trở thành một hàng số. Cột `Benh` là nhãn cần dự đoán.


In [4]:
# Các cột chứa triệu chứng trong file gốc
cot_trieuchung = [c for c in benh.columns if c.startswith("TrieuChung")]

# Tạo bảng toàn số 0, mỗi cột là một triệu chứng
X = pd.DataFrame(0, index=benh.index, columns=ds_trieuchung)

# Duyệt từng ca bệnh, điền trọng số vào đúng cột
for i in range(len(benh)):
    for cot in cot_trieuchung:
        tc = benh.loc[i, cot]
        if isinstance(tc, str) and tc.strip() in trong_so:
            X.loc[i, tc.strip()] = trong_so[tc.strip()]

# Gắn cột nhãn
X["Benh"] = benh["Benh"]
print("Bảng đặc trưng:", X.shape)
X.head()


Bảng đặc trưng: (2100, 38)


,sốt cao,sốt,sốt nhẹ,đau đầu,đau nhức cơ,mệt mỏi,ho,ho có đờm,đau họng,ớn lạnh,...,chán ăn,đổ mồ hôi,chảy nước mắt,ngứa,nhạy cảm ánh sáng,chóng mặt,đau mặt,giảm khứu giác,thở khò khè,Benh
0,6,0,0,0,0,3,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Cảm cúm
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,5,Hen suyễn
2,0,0,0,0,0,0,0,0,4,0,...,0,0,0,0,0,0,0,0,0,Viêm họng
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Viêm dạ dày
4,0,0,0,0,0,0,3,0,0,0,...,0,0,0,0,0,0,0,0,5,Hen suyễn


## Bước 4. Lưu kết quả ra file

In [5]:
import os
os.makedirs("../01_data/processed", exist_ok=True)
X.to_csv("../01_data/processed/features.csv", index=False, encoding="utf-8-sig")
print("Đã lưu: 01_data/processed/features.csv")


Đã lưu: 01_data/processed/features.csv
